# E07 · Run OWL DL and understand its limits

**Outcome:** Classify with HermiT, inspect profile reports, and distinguish graph answers from entailment.

**Time:** about 55 minutes. Run cells in order. Edit the exercise cell after completing the walkthrough.

OWL 2 DL reasoning is model-theoretic: an entailment holds in every interpretation satisfying the ontology. It is not a traversal of a few subclass edges. The imported BFO core and existential restrictions in this course require a DL-capable path. The RL exercise remains useful on a separately restricted module; owlrl is not a replacement for HermiT on this input.

Run profile validation before classification and consistency before returning answers. An unsatisfiable class cannot have members in any model but an ontology containing an empty unsatisfiable class can remain consistent. Asserting an instance of that class makes the ontology inconsistent. Different names do not automatically denote different individuals. A maximum-cardinality restriction can force equality, whereas an explicit inequality can reveal a contradiction.

Our exporter asks HermiT for named class memberships of the named individuals. It does not implement a complete SPARQL OWL Direct Semantics endpoint. Arbitrary joins over unnamed existential witnesses, negative facts and inferred object-property values are outside the exported query contract. SPARQL over the selected export is useful precisely because its coverage is stated. Count returned named records, not all possible objects in a model.

For an existential restriction, a reasoner may conclude that a suitable object exists without assigning it a public IRI. A missing documents edge can therefore be consistent in OWL while failing the SHACL requirement that the edge be supplied. Do not manufacture a named witness merely to make an export look complete.

In [1]:
from pathlib import Path
import sys, json
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "ontology_lab").is_dir():
        sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError("Open this notebook from the extracted course folder.")
from ontology_lab import *
print("Course:", ROOT.name, "| source rows:", len(rows()))

Course: enterprise_ontology_tutorial | source rows: 60


## Read the profile and class results

In [2]:
report,inferred=reasoning()
display({k:report[k] for k in ['engine','scope','consistent','unsatisfiable','export_coverage']})
assert report['DL']['in_profile'] and not report['RL']['in_profile']
assert len(members(inferred,EX.CardiopulmonaryRecord))==40

{'engine': 'HermiT via Owlready2 0.51 / OWLAPI',
 'scope': 'Complete pinned BFO core + clinical schema + supplied ABox',
 'consistent': True,
 'unsatisfiable': [],
 'export_coverage': 'Named class memberships of named individuals only'}

## Remove a required explicit edge in a local experiment

In [3]:
changed=build_asserted()
record=record_iri(rows()[0])
changed.remove((record,EX.documents,None))
dl_report,_=reasoning(source=changed)
print('OWL consistent:',dl_report['consistent'])
print('SHACL conforms:',validate(changed)[0])
assert dl_report['consistent'] and not validate(changed)[0]

OWL consistent: True
SHACL conforms: False


## Compare a restricted RL module

In [4]:
from owlrl import DeductiveClosure, OWLRL_Semantics
rl=Graph()
rl.add((EX.ReviewCandidate,RDF.type,OWL.Class))
rl.add((EX.EncounterRecord,RDF.type,OWL.Class))
rl.add((EX.ReviewCandidate,RDFS.subClassOf,EX.EncounterRecord))
record=record_iri(next(r for r in rows() if r['diag_1']=='493'))
rl.add((record,RDF.type,EX.ReviewCandidate))
assert (record,RDF.type,EX.EncounterRecord) not in rl
DeductiveClosure(OWLRL_Semantics).expand(rl)
assert (record,RDF.type,EX.EncounterRecord) in rl
print('RL derives the superclass in this restricted module; it does not replace the full DL run.')

RL derives the superclass in this restricted module; it does not replace the full DL run.


## Your turn

Return the number of named cardiopulmonary-coded records entailed by the supplied ontology.

Replace `answer = None` with your code. A skipped exercise is reported as incomplete; it is not a pass.

In [5]:
answer = None  # Write your solution here

In [6]:
learner_check(answer, lambda x: x == 40, 'Use the named class memberships returned by the reasoner.')

Exercise not completed. Use the named class memberships returned by the reasoner.
Out[0]: False


## Explain your model

Why is absence from a SPARQL result not an OWL proof of negation?

Write a short answer below. Check the relevant chapter in the book before promoting a model change.

**My explanation:** _Write your explanation here._